[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [21]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [20]:
import torch
import torch.nn as nn
import math

In [33]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # pass  # W_q, W_k, W_v, W_o
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv):
        # pass  # Q from x_q, K/V from x_kv, no causal mask

        B, S_q, D = x_q.shape
        _, S_kv, _ = x_kv.shape

        print(f"x_q shape: {x_q.shape}")
        print(f"x_kv shape: {x_kv.shape}\n\n")


        # (B, Q, D)
        Q = self.W_q(x_q)
        K = self.W_k(x_kv)
        V = self.W_k(x_kv)

        print(f"Q shape: {Q.shape}")
        print(f"K shape: {K.shape}")
        print(f"V shape: {V.shape}\n\n")

        Q = Q.view(B, S_q, self.num_heads, self.head_dim)
        K = K.view(B, S_kv, self.num_heads, self.head_dim)
        V = V.view(B, S_kv, self.num_heads, self.head_dim)

        # (B, Q, H, D)
        print(f"Q shape: {Q.shape}")
        print(f"K shape: {K.shape}")
        print(f"V shape: {V.shape}\n\n")

        Q = Q.transpose(1,2)
        K = K.transpose(1,2)
        V = V.transpose(1,2)

        # (B, H, Q, D)
        print(f"Q shape: {Q.shape}")
        # (B, H, K, D)
        print(f"K shape: {K.shape}")
        print(f"V shape: {V.shape}\n\n")

        # (B, H, D, K)
        K = K.transpose(-2,-1)
        print(f"K shape: {K.shape}")


        d_k = Q.size(-1)
        # (B, H, Q, D) @ (B, H, D, K) = (B, H, Q, K)
        scores = torch.matmul(Q, K) / math.sqrt(d_k)
        # (B, H, Q, K)
        print(f"scores shape: {scores.shape}")
        weights = torch.softmax(scores, dim=-1)
        print(f"weights shape: {weights.shape}")

        # (B, H, Q, K) @ (B, H, K, D) = (B, H, Q, D)
        attn_output = torch.matmul(weights, V)
        print(f"attn_output shape: {attn_output.shape}\n\n")


        # (B, Q, H, D)
        x = attn_output.transpose(1,2).contiguous()
        print(f"transposed attn_output shape: {x.shape}")

        # (B, Q, D)
        x = x.view(B, S_q, self.d_model)
        print(f"concat attn_output shape: {x.shape}")

        return self.W_o(x)



In [34]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

x_q shape: torch.Size([2, 6, 64])
x_kv shape: torch.Size([2, 10, 64])


Q shape: torch.Size([2, 6, 64])
K shape: torch.Size([2, 10, 64])
V shape: torch.Size([2, 10, 64])


Q shape: torch.Size([2, 6, 4, 16])
K shape: torch.Size([2, 10, 4, 16])
V shape: torch.Size([2, 10, 4, 16])


Q shape: torch.Size([2, 4, 6, 16])
K shape: torch.Size([2, 4, 10, 16])
V shape: torch.Size([2, 4, 10, 16])


K shape: torch.Size([2, 4, 16, 10])
scores shape: torch.Size([2, 4, 6, 10])
weights shape: torch.Size([2, 4, 6, 10])
attn_output shape: torch.Size([2, 4, 6, 16])


transposed attn_output shape: torch.Size([2, 6, 4, 16])
concat attn_output shape: torch.Size([2, 6, 64])
Output: torch.Size([2, 6, 64])


In [35]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
x_q shape: torch.Size([2, 6, 64])
x_kv shape: torch.Size([2, 10, 64])


Q shape: torch.Size([2, 6, 64])
K shape: torch.Size([2, 10, 64])
V shape: torch.Size([2, 10, 64])


Q shape: torch.Size([2, 6, 4, 16])
K shape: torch.Size([2, 10, 4, 16])
V shape: torch.Size([2, 10, 4, 16])


Q shape: torch.Size([2, 4, 6, 16])
K shape: torch.Size([2, 4, 10, 16])
V shape: torch.Size([2, 4, 10, 16])


K shape: torch.Size([2, 4, 16, 10])
scores shape: torch.Size([2, 4, 6, 10])
weights shape: torch.Size([2, 4, 6, 10])
attn_output shape: torch.Size([2, 4, 6, 16])


transposed attn_output shape: torch.Size([2, 6, 4, 16])
concat attn_output shape: torch.Size([2, 6, 64])
  ✅ [1/4] Output shape (1.9ms)
x_q shape: torch.Size([1, 3, 32])
x_kv shape: torch.Size([1, 20, 32])


Q shape: torch.Size([1, 3, 32])
K shape: torch.Size([1, 20, 32])
V shape: torch.Size([1, 20, 32])


Q shape: torch.Size([1, 3, 2, 16])
K sh